compares model performance across models trained with train_models.ipynb. 

In [2]:
import xarray as xr
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import KFold
from torch.utils.data import DataLoader
import os
import pandas as pd
import matplotlib.pyplot as plt

from src.dataset import LazyWeatherDataset
from src.preprocessing import flatten_target_dataset, standardize_with_stats, compute_overall_from_daily_stats
from src.train_loop import evaluate
from src.models import get_model, get_model_input_dims

Errors for raw outlooks

TODO: 
For each specified model: (following train framework?)
    - load in requested model, training inputs and targets
    - Evaluate and grab all 12 MSEs at the best iteration? Or final iteration? probably final--that way we can do from already trained models
    - Average over all 5 folds
    - sqrt to Rrmse_units then convert to original units
    - save to results?
When back on ncar, do with all models.
Visualize errors


In [2]:
target_str = '_new'

In [3]:
def evaluate_model(X, y, stats, model_name, n_splits=5, batch_size=64, optimizer_class=torch.optim.Adam, lr=1e-3, criterion=nn.MSELoss(), level=None, latest = True):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    days = X.day.values
    kf = KFold(n_splits=n_splits, shuffle=False)
    overall_stats = compute_overall_from_daily_stats(stats)
    val_counts = []

    losses = []
    predictions = []
    targets = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(days)):

        val_counts.append(val_idx.shape[0])

        model_spec = f"{model_name}/level={level}/opt={optimizer_class.__name__}_lr={lr}_batch={batch_size}_crit={criterion.__class__.__name__}/fold={fold}"

        model_dict = f"models/{model_spec}/"

        print(f"Fold {fold}")

        # print("Selecting data...")
        train_days = days[train_idx]
        val_days = days[val_idx]

        X_train = X.sel(day=train_days)
        X_val = X.sel(day=val_days)

        y_train = y.sel(time=train_days)
        y_val = y.sel(time=val_days)

        # print("Computing Stats data...")
        fold_stats = compute_overall_from_daily_stats(stats.sel(day=train_days))

        # Dummy means and std to "restandardize" the data with.
        # Essentially, the subsequent standardize() will, for both the current training and validation sets
        # 1) unstandardize (since we start with data that has been standardized across the entire non-test dataset), then
        # 2) standardize the current fold's training and validation data according to the mean and std of the current fold's training set
        # print("Standardizing data...")
        conversion_stats = xr.Dataset({
            v: ((fold_stats[v] - overall_stats[v]) / overall_stats[v.replace('_mean', '_std')]) if v.endswith('_mean') else (fold_stats[v] / overall_stats[v])
            for v in fold_stats.data_vars
        })

        X_train_standardized = standardize_with_stats(X_train, conversion_stats)
        X_val_standardized = standardize_with_stats(X_val, conversion_stats)

        # print("Setting up datasets...")
        input_dimensions = get_model_input_dims(model_name)

        train_ds = LazyWeatherDataset(X_train_standardized, y=flatten_target_dataset(y_train), input_dimensions=input_dimensions)
        val_ds = LazyWeatherDataset(X_val_standardized, y=flatten_target_dataset(y_val), input_dimensions=input_dimensions)

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=batch_size)

        # ==== Model setup ====
        x_example, y_example = next(iter(train_loader))
        input_dim = x_example.shape[1:] if x_example.ndim > 2 else x_example.shape[1]
        output_dim = y_example.shape[1] if y_example.ndim > 1 else 1

        t = None

        model = get_model(model_name, input_dim, output_dim, t).to(device)

        if latest:
            latest_path = os.path.join(model_dict, "latest.pt")
        else:
            latest_path = os.path.join(model_dict, "best.pt")
        # print('loading model')
        model.load_state_dict(torch.load(latest_path, map_location=torch.device('cpu'))['model_state_dict'])
        # print("evaluating model")
        val_loss, all_preds, all_targets = evaluate(model, val_loader, criterion, device, None, None)
        losses.append(val_loss)
        predictions.append(all_preds)
        targets.append(all_targets)
    return val_counts, losses, predictions, targets, model_spec.split('/fold')[0]

In [ ]:
latest = False

if latest:
    latest_str = 'latest_'
else:
    latest_str = 'best_'

input_dir = "/glade/work/milesep/convective_outlook_ml"
target_dir = "data/processed_data"
stats_dir = "data/processed_data"

mse_path = "results/" + latest_str + "all_mse.csv"
rmse_path = "results/" + latest_str + "all_rmses.csv"
rmse_units_path = "results/" + latest_str + "all_rmses_units.csv"

# model_names = ["cnn3d_3_layer", "cnn3d_dropout_5_0", "cnn3d_dropout_5_0", "cnn3d_dropout_5_5", "cnn3d_dropout_5_5"]
# levels = ["small", "small", "small", "small", "small"]
# lrs = [1e-3, 1e-3, 1e-3, 1e-3, 1e-3]
# batch_sizes = [8, 32, 64, 32, 64]

# for name, level, lr, batch_size in zip(model_names, levels, lrs, batch_sizes):

results_df = pd.read_csv('results/results.csv')
model_names = results_df['model']

if os.path.exists(mse_path):
    mse_df = pd.read_csv(mse_path)
    done_model_specs = mse_df['model']
else:
    done_model_specs = []
id = 0
for model in model_names:
    # print(model)
    id += 1
    if 'batch' in model and not (model.split('/')[0] in ['predict_mean', 'predict_zero']) and not (model in list(done_model_specs)):
        name = model.split('/')[0]
        level = model.split('level=')[1].split('/')[0]
        lr = float(model.split('lr=')[1].split('_')[0])
        batch_size = int(model.split('batch=')[1].split('_')[0])
        if level[:4] == 'slgt':
            slgt_mod_str = '_slgt'
        else:
            slgt_mod_str = ''
        print(id, name, level, lr, batch_size)
        if True:
            # print("LEVEL!!!!!" + level)
            lev = level.removesuffix("_new")
            inputs = xr.open_zarr(f"{input_dir}/train_inputs_{lev}.zarr")
            if level.endswith("_new"):
                tars = xr.open_dataset(f"{target_dir}/train_targets{slgt_mod_str}_new.nc")
            else:
                tars = xr.open_dataset(f"{target_dir}/train_targets{slgt_mod_str}.nc")
            stats = xr.open_dataset(f"{stats_dir}/daily_input_stats_{lev}.nc")
            sizes, losses, predictions, targets, model_spec = evaluate_model(inputs, tars, stats, name, batch_size = batch_size, lr = lr, level = level, latest = latest)
    
            all_preds = np.array(predictions[0])
            all_targets = np.array(targets[0])
    
            for i in range(1, len(predictions)):
                all_preds = np.append(all_preds, np.array(predictions[i]), axis = 0)
                all_targets = np.append(all_targets, np.array(targets[i]), axis = 0)
    
            mses = ((all_preds - all_targets)**2).mean(axis=0)
            rmses = np.sqrt(mses)
            stds = tars['train_std'].transpose("hazard", "variable").values.flatten()
            rmse_units = rmses * stds
    
            mse_row = {
                "model": model_spec,
                **{str(i): v for i, v in enumerate(mses)}
            }
    
            rmse_row = {
                "model": model_spec,
                **{str(i): v for i, v in enumerate(rmses)}
            }
    
            rmse_units_row = {
                "model": model_spec,
                **{str(i): v for i, v in enumerate(rmse_units)}
            }
    
            os.makedirs(os.path.dirname(mse_path), exist_ok=True)
            os.makedirs(os.path.dirname(rmse_path), exist_ok=True)
            os.makedirs(os.path.dirname(rmse_units_path), exist_ok=True)
    
            if os.path.exists(mse_path):
                df = pd.read_csv(mse_path)
                df = df[df["model"] != mse_row["model"]]  # overwrite if it exists
                df = pd.concat([df, pd.DataFrame([mse_row])], ignore_index=True)
            else:
                df = pd.DataFrame([mse_row])
    
            df.to_csv(mse_path, index=False)
    
            if os.path.exists(rmse_path):
                df = pd.read_csv(rmse_path)
                df = df[df["model"] != rmse_row["model"]]  # overwrite if it exists
                df = pd.concat([df, pd.DataFrame([rmse_row])], ignore_index=True)
            else:
                df = pd.DataFrame([rmse_row])
    
            df.to_csv(rmse_path, index=False)
    
            if os.path.exists(rmse_units_path):
                df = pd.read_csv(rmse_units_path)
                df = df[df["model"] != rmse_units_row["model"]]  # overwrite if it exists
                df = pd.concat([df, pd.DataFrame([rmse_units_row])], ignore_index=True)
            else:
                df = pd.DataFrame([rmse_units_row])
    
            df.to_csv(rmse_units_path, index=False)

Run the following 2 cells to cover both slgt = True and slgt = False. Can skip if it's been run before. Also rerun with latest true/false, target_str '' and 'new' to cover all bases

In [4]:
target_str = '_new'
slgt = True
if slgt:
    mod_str = '_slgt'
else:
    mod_str = ''

latest = False
if latest:
    latest_str = 'latest_'
else:
    latest_str = 'best_'

zero_targets = xr.open_dataset(f"data/processed_data/train_targets_zero{mod_str}{target_str}.nc")

target_dir = "data/processed_data"
targets = xr.open_dataset(f"{target_dir}/train_targets{mod_str}{target_str}.nc")
targets = targets.drop_vars(["train_mean", "train_std"], errors="ignore")
zero_targets = zero_targets.drop_vars(["train_mean", "train_std"], errors="ignore")

# TODO: also do with learned means instead of doing zeros_like, although test showed difference is negligible
diff = []
diff1 = []
for v in targets.data_vars:
    d = (targets[v] - zero_targets[v])**2
    diff.append(d)
    d1 = (targets[v] - np.zeros_like(zero_targets[v]))**2
    diff1.append(d1)

# stack into one DataArray
all_diffs = xr.concat(diff, dim="variable2")
all_diffs1 = xr.concat(diff1, dim="variable2")

C:\Users\miles\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\xarray\core\concat.py:532: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  common_dims = tuple(pd.unique([d for v in vars for d in v.dims]))


In [5]:
all_diffs

<xarray.DataArray 'bias' (variable2: 3, hazard: 3, time: 3414)>
array([[[4.09304796e-03, 6.24073798e-02, 3.24557356e-01, ...,
         3.34126364e-04, 2.90773468e-01, 5.94002425e-04],
        [2.22142427e-01, 3.62295895e-02, 9.23074077e-01, ...,
         0.00000000e+00, 2.35143181e-01, 0.00000000e+00],
        [0.00000000e+00, 7.52806197e-04, 1.49665400e-01, ...,
         3.74362994e-01, 1.17249866e+01, 6.33110011e-03]],

       [[1.33852789e-02, 2.93526730e+00, 3.98925687e-01, ...,
         1.20138732e-01, 9.14652415e-01, 8.33740137e-03],
        [7.36963011e-01, 5.95213355e-03, 2.69858490e+00, ...,
         0.00000000e+00, 5.55907836e-01, 0.00000000e+00],
        [0.00000000e+00, 2.16738701e-01, 2.63885301e-04, ...,
         4.05400194e-01, 7.72875521e-01, 2.70130342e-03]],

       [[3.14908899e-02, 7.41780737e-01, 1.25608718e+01, ...,
         1.86114974e-01, 9.83043271e-01, 2.95696904e-03],
        [6.33014567e-03, 8.13686308e-02, 4.49437043e+00, ...,
         0.00000000e+00, 4.60071619e-03, 0.00000000e+00],
        [0.00000000e+00, 1.46789569e+00, 1.20754424e-02, ...,
         2.18334769e-01, 1.76671709e+00, 1.86408629e-03]]])
Coordinates:
  * time     (time) datetime64[ns] 2002-04-02 2002-04-03 ... 2019-12-22
  * hazard   (hazard) object 'Wind' 'Hail' 'Tornado'
Dimensions without coordinates: variable2

In [ ]:
slgt = True
if slgt:
    mod_str = '_slgt'
else:
    mod_str = ''

latest = False
if latest:
    latest_str = 'latest_'
else:
    latest_str = 'best_'

zero_targets = xr.open_dataset(f"data/processed_data/train_targets_zero{mod_str}{target_str}.nc")

target_dir = "data/processed_data"
targets = xr.open_dataset(f"{target_dir}/train_targets{mod_str}{target_str}.nc")
targets = targets.drop_vars(["train_mean", "train_std"], errors="ignore")
zero_targets = zero_targets.drop_vars(["train_mean", "train_std"], errors="ignore")

# TODO: also do with learned means instead of doing zeros_like, although test showed difference is negligible
diff = []
diff1 = []
for v in targets.data_vars:
    d = (targets[v] - zero_targets[v])**2
    diff.append(d)
    d1 = (targets[v] - np.zeros_like(zero_targets[v]))**2
    diff1.append(d1)

# stack into one DataArray
all_diffs = xr.concat(diff, dim="variable2")
all_diffs1 = xr.concat(diff1, dim="variable2")
raw_mses = all_diffs.mean(dim = 'time').transpose("hazard", "variable2").values.flatten()
zero_mses = all_diffs1.mean(dim = 'time').transpose("hazard", "variable2").values.flatten()

mse_path = "results/" + latest_str + "all_mse.csv"
rmse_path = "results/" + latest_str + "all_rmses.csv"
rmse_units_path = "results/" + latest_str + "all_rmses_units.csv"

tars = xr.open_dataset(f"{target_dir}/train_targets{mod_str}{target_str}.nc")

for mses, model_spec in zip([raw_mses, zero_mses], ['raw_outlook' + mod_str + target_str, 'predict_zero' + mod_str + target_str]):
    rmses = np.sqrt(mses)
    stds = tars['train_std'].transpose("hazard", "variable").values.flatten()
    rmse_units = rmses * stds

    mse_row = {
        "model": model_spec,
        **{str(i): v for i, v in enumerate(mses)}
    }

    rmse_row = {
        "model": model_spec,
        **{str(i): v for i, v in enumerate(rmses)}
    }

    rmse_units_row = {
        "model": model_spec,
        **{str(i): v for i, v in enumerate(rmse_units)}
    }

    os.makedirs(os.path.dirname(mse_path), exist_ok=True)
    os.makedirs(os.path.dirname(rmse_path), exist_ok=True)
    os.makedirs(os.path.dirname(rmse_units_path), exist_ok=True)

    if os.path.exists(mse_path):
        df = pd.read_csv(mse_path)
        df = df[df["model"] != mse_row["model"]]  # overwrite if it exists
        df = pd.concat([df, pd.DataFrame([mse_row])], ignore_index=True)
    else:
        df = pd.DataFrame([mse_row])

    df.to_csv(mse_path, index=False)

    if os.path.exists(rmse_path):
        df = pd.read_csv(rmse_path)
        df = df[df["model"] != rmse_row["model"]]  # overwrite if it exists
        df = pd.concat([df, pd.DataFrame([rmse_row])], ignore_index=True)
    else:
        df = pd.DataFrame([rmse_row])

    df.to_csv(rmse_path, index=False)

    if os.path.exists(rmse_units_path):
        df = pd.read_csv(rmse_units_path)
        df = df[df["model"] != rmse_units_row["model"]]  # overwrite if it exists
        df = pd.concat([df, pd.DataFrame([rmse_units_row])], ignore_index=True)
    else:
        df = pd.DataFrame([rmse_units_row])

    df.to_csv(rmse_units_path, index=False)

In [ ]:
slgt = False
if slgt:
    mod_str = '_slgt'
else:
    mod_str = ''

latest = False
if latest:
    latest_str = 'latest_'
else:
    latest_str = 'best_'

zero_targets = xr.open_dataset(f"data/processed_data/train_targets_zero{mod_str}{target_str}.nc")

target_dir = "data/processed_data"
targets = xr.open_dataset(f"{target_dir}/train_targets{mod_str}{target_str}.nc")
targets = targets.drop_vars(["train_mean", "train_std"], errors="ignore")
zero_targets = zero_targets.drop_vars(["train_mean", "train_std"], errors="ignore")

# TODO: also do with learned means instead of doing zeros_like, although test showed difference is negligible
diff = []
diff1 = []
for v in targets.data_vars:
    d = (targets[v] - zero_targets[v])**2
    diff.append(d)
    d1 = (targets[v] - np.zeros_like(zero_targets[v]))**2
    diff1.append(d1)

# stack into one DataArray
all_diffs = xr.concat(diff, dim="variable2")
all_diffs1 = xr.concat(diff1, dim="variable2")
raw_mses = all_diffs.mean(dim = 'time').transpose("hazard", "variable2").values.flatten()
zero_mses = all_diffs1.mean(dim = 'time').transpose("hazard", "variable2").values.flatten()

mse_path = "results/" + latest_str + "all_mse.csv"
rmse_path = "results/" + latest_str + "all_rmses.csv"
rmse_units_path = "results/" + latest_str + "all_rmses_units.csv"

tars = xr.open_dataset(f"{target_dir}/train_targets{mod_str}{target_str}.nc")

for mses, model_spec in zip([raw_mses, zero_mses], ['raw_outlook' + mod_str + target_str, 'predict_zero' + mod_str + target_str]):
    rmses = np.sqrt(mses)
    stds = tars['train_std'].transpose("hazard", "variable").values.flatten()
    rmse_units = rmses * stds

    mse_row = {
        "model": model_spec,
        **{str(i): v for i, v in enumerate(mses)}
    }

    rmse_row = {
        "model": model_spec,
        **{str(i): v for i, v in enumerate(rmses)}
    }

    rmse_units_row = {
        "model": model_spec,
        **{str(i): v for i, v in enumerate(rmse_units)}
    }

    os.makedirs(os.path.dirname(mse_path), exist_ok=True)
    os.makedirs(os.path.dirname(rmse_path), exist_ok=True)
    os.makedirs(os.path.dirname(rmse_units_path), exist_ok=True)

    if os.path.exists(mse_path):
        df = pd.read_csv(mse_path)
        df = df[df["model"] != mse_row["model"]]  # overwrite if it exists
        df = pd.concat([df, pd.DataFrame([mse_row])], ignore_index=True)
    else:
        df = pd.DataFrame([mse_row])

    df.to_csv(mse_path, index=False)

    if os.path.exists(rmse_path):
        df = pd.read_csv(rmse_path)
        df = df[df["model"] != rmse_row["model"]]  # overwrite if it exists
        df = pd.concat([df, pd.DataFrame([rmse_row])], ignore_index=True)
    else:
        df = pd.DataFrame([rmse_row])

    df.to_csv(rmse_path, index=False)

    if os.path.exists(rmse_units_path):
        df = pd.read_csv(rmse_units_path)
        df = df[df["model"] != rmse_units_row["model"]]  # overwrite if it exists
        df = pd.concat([df, pd.DataFrame([rmse_units_row])], ignore_index=True)
    else:
        df = pd.DataFrame([rmse_units_row])

    df.to_csv(rmse_units_path, index=False)

Plot!

In [6]:
latest = False

if latest:
    latest_str = 'latest_'
else:
    latest_str = 'best_'

mse_path = "results/" + latest_str + "all_mse.csv"
rmse_path = "results/" + latest_str + "all_rmses.csv"
rmse_units_path = "results/" + latest_str + "all_rmses_units.csv"

rmses_units = pd.read_csv(rmse_units_path)

In [7]:
# Define helper to pick baseline rows
def get_baselines(df, model_name, target_str):
    if "slgt" in model_name:
        raw_row = df[df["model"] == ("raw_outlook_slgt" + target_str)].iloc[0]
        mean_row = df[df["model"] == ("predict_zero_slgt" + target_str)].iloc[0]
    else:
        raw_row = df[df["model"] == ("raw_outlook" + target_str)].iloc[0]
        mean_row = df[df["model"] == ("predict_zero" + target_str)].iloc[0]
    return raw_row, mean_row


# Rows to skip
skip_models = {"raw_outlook", "predict_zero", "raw_outlook_slgt", "predict_zero_slgt", "raw_outlook_new", "predict_zero_new", "raw_outlook_slgt_new", "predict_zero_slgt_new"}

for idx, row in rmses_units.iterrows():
    model_name = row["model"]
    print(model_name)
    if '_new' in model_name:
        target_str = '_new'
    else:
        target_str = ''

    if target_str == '':
        hazard_labels = ['All Hazard', 'Wind', 'Hail', 'Tornado']
    elif target_str == '_new':
        hazard_labels = ['Wind', 'Hail', 'Tornado']
    x = np.arange(len(hazard_labels))
    width = 0.6
    if model_name in skip_models:
        continue

    model_row = row
    raw_row, mean_row = get_baselines(rmses_units, model_name, target_str)

    for bias_type in range(3):
        raw_vals, mean_vals, model_vals = [], [], []

        for h in range(len(hazard_labels)):
            col = str(bias_type*len(hazard_labels) + h)

            def _get_val(r, col_name):
                if col_name in r.index:
                    return float(r[col_name])
                return float(r.iloc[int(col_name)])

            raw_vals.append(_get_val(raw_row, col))
            mean_vals.append(_get_val(mean_row, col))
            model_vals.append(_get_val(model_row, col))

        if bias_type == 0:
            div = 1
        else:
            div = 1000

        raw_vals = np.array(raw_vals)/div
        mean_vals = np.array(mean_vals)/div
        model_vals = np.array(model_vals)/div

        fig, ax = plt.subplots(figsize=(2.5*len(hazard_labels), 6))

        # Bar widths
        raw_width = width
        mean_width = width * 0.7
        model_width = width * 0.45

        raw_x = x
        mean_x = x - (mean_width - raw_width)/2.0
        model_x = x - (model_width - raw_width)/2.0

        # Draw bars
        ax.bar(raw_x, raw_vals, raw_width, label='Raw Outlook Bias', color='C0', alpha=0.9, zorder=1)
        ax.bar(mean_x, mean_vals, mean_width, label='Applied Mean Bias Correction', color='C1', alpha=0.9, zorder=2)
        ax.bar(model_x, model_vals, model_width, label='Applied Bias Correction by Model', color='C2', alpha=0.95, zorder=3)

        ax.set_xticks(x)
        ax.set_xticklabels(hazard_labels)

        if bias_type == 0:
            ax.set_ylabel('Count Bias (Number of Gridpoints)')
        elif bias_type == 1:
            ax.set_ylabel('E-W Displacement (km)')
        else:
            ax.set_ylabel('N-S Displacement (km)')

        bias_titles = ['RMS Count Bias', 'RMS E-W Displacement Bias', 'RMS N-S Displacement Bias']
        ax.set_title(f"{bias_titles[bias_type]}")
 
        ax.legend()
        plt.tight_layout()

        bias_type_dict = {0: 'count',
                          1: 'e-w',
                          2: 'n-s'}

        if 'slgt_full' in model_name:
            folder = 'slgt_full'
        elif 'full' in model_name:
            folder = 'mdt_full'
        elif 'slgt' in model_name:
            folder = 'slgt'
        else:
            folder = 'mdt'
        # Save fig
        safe_name = model_name.replace("/", "_")
        if latest:
            latest_folder = 'latest'
        else:
            latest_folder = 'best'
        # Ensure output dir exists
        os.makedirs(f"figs/error/{latest_folder}/{folder}/{safe_name}{target_str}", exist_ok=True)
        plt.savefig(f"figs/error/{latest_folder}/{folder}/{safe_name}{target_str}/{bias_type_dict[bias_type]}.png")
        # plt.show()
        plt.close(fig)

linear_regression/level=small/opt=Adam_lr=0.001_batch=32_crit=MSELoss
cnn3d_dropout_0_5/level=small/opt=Adam_lr=0.001_batch=32_crit=MSELoss
cnn3d_dropout_5_5/level=small/opt=Adam_lr=0.001_batch=32_crit=MSELoss
cnn3d_dropout_5_0/level=small/opt=Adam_lr=0.001_batch=32_crit=MSELoss
cnn3d/level=small/opt=Adam_lr=0.001_batch=32_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.001_batch=4_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.001_batch=8_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.001_batch=32_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.0001_batch=4_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.0001_batch=8_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.0001_batch=32_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.0004_batch=4_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.0004_batch=8_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.0004_batch=32_crit=MSELoss
cnn3d_gelu_0_5/level=small/opt=Adam_lr=0.001_batch=64_crit=